In [1]:
from main import run_bot
from demo import run_demo
import config

In [2]:
from config import MIN_R2, OUTPUTS_DIR
from broker.connection import connect_ib
from broker.data import fetch_prices, fetch_prices_free
from broker.orders import calculate_position_size, execute_order, get_portfolio_value
from analysis.universe import fetch_market_caps, get_sp500_tickers
from analysis.correlations import compute_correlations, get_top_correlated_pairs, get_top_inverse_pairs
from analysis.model import predict_price
from analysis.signals import generate_signals
from reporting.charts import (plot_correlation_matrix, plot_market_cap_bars,
                               plot_prediction_analysis, plot_price_series)
from reporting.report import print_report, save_signals_csv

In [3]:
n_tickers = None    # int = top N by market cap | None = full S&P 500 | 'FALLBACK_TICKERS' = hardcoded top-20
mode = 'paper'      # demo | paper | live | signals
execute_trades=True

In [4]:
print("\nFetching S&P 500 universe...")
tickers, market_caps = get_sp500_tickers(n=n_tickers)


Fetching S&P 500 universe...
  ✓ 503 tickers fetched from Wikipedia
  Sorting 503 tickers by market cap via yfinance (this takes ~30s)...

Fetching market caps for 503 tickers via yfinance...
  ✓ Market caps retrieved: 503/503
  → Using all 503 S&P 500 tickers


In [6]:
prices_df = fetch_prices_free(tickers)
if prices_df.empty or len(prices_df.columns) < 5:
    print("✗ Insufficient data. Aborting.")
    return


  Tickers with data: 503/503
  ✓ A: 800 bars — last close $118.05
  ✓ AAPL: 800 bars — last close $288.92
  ✓ ABBV: 800 bars — last close $203.25
  ✓ ABNB: 800 bars — last close $139.88
  ✓ ABT: 800 bars — last close $87.54
  ✓ ACGL: 800 bars — last close $94.88
  ✓ ACN: 800 bars — last close $178.86
  ✓ ADBE: 800 bars — last close $257.11
  ✓ ADI: 800 bars — last close $406.32
  ✓ ADM: 800 bars — last close $77.76
  ✓ ADP: 800 bars — last close $214.28
  ✓ ADSK: 800 bars — last close $252.17
  ✓ AEE: 800 bars — last close $108.71
  ✓ AEP: 800 bars — last close $132.54
  ✓ AES: 800 bars — last close $14.35
  ✓ AFL: 800 bars — last close $113.33
  ✓ AIG: 800 bars — last close $76.83
  ✓ AIZ: 800 bars — last close $235.17
  ✓ AJG: 800 bars — last close $202.55
  ✓ AKAM: 800 bars — last close $112.12
  ✓ ALB: 800 bars — last close $202.71
  ✓ ALGN: 800 bars — last close $169.24
  ✓ ALL: 800 bars — last close $214.45
  ✓ ALLE: 800 bars — last close $136.21
  ✓ AMAT: 800 bars — last close 

SyntaxError: 'return' outside function (1033143228.py, line 4)

In [ ]:
print("\nCalculating correlations...")
corr_matrix, returns = compute_correlations(prices_df)
top_pairs     = get_top_correlated_pairs(corr_matrix, top_n=10)
inverse_pairs = get_top_inverse_pairs(corr_matrix, top_n=10)

In [ ]:
signals_df = generate_signals(prices_df, returns, corr_matrix)
print_report(signals_df, top_pairs, inverse_pairs)
save_signals_csv(signals_df)

In [ ]:
save_plots = 1
if save_plots:
    plot_correlation_matrix(corr_matrix)

    # All tickers in gray, top 15 most valuable in color
    plot_price_series(prices_df, tickers, top_n=15)

    # Top 15 vs bottom 15 — stock price (top) and market cap (bottom)
    plot_market_cap_bars(prices_df, tickers, market_caps=market_caps, top_n=15)

    # Dual-subplot analysis for top 5 signals by predicted return
    top_signals = signals_df.head(5)
    if not top_signals.empty:
        print("\nGenerating analysis charts...")
    for _, row in top_signals.iterrows():
        ticker = row['ticker']
        pred_ret, r2, top5, corr_signs, y_actual, y_pred = predict_price(
            ticker, returns, corr_matrix
        )
        if y_actual is not None:
            plot_prediction_analysis(
                ticker, returns, prices_df, top5, corr_signs,
                y_actual, y_pred,
                save_path=OUTPUTS_DIR / f'analysis_{ticker}.png'
            )

In [ ]:
if execute_trades:
    ib = connect_ib()
    try:
        portfolio_value = get_portfolio_value(ib)
        print(f"\nPlacing orders (portfolio: ${portfolio_value:,.0f})...")
        actionable = signals_df[signals_df['signal'].isin(['BUY', 'SELL'])]
        for _, row in actionable.iterrows():
            if row['model_r2'] < MIN_R2:
                continue
            strength = min(1.0, row['model_r2'])
            qty = calculate_position_size(portfolio_value, row['current_price'], strength)
            execute_order(ib, row['ticker'], row['signal'], qty)
    finally:
        ib.disconnect()
        print("\n✓ Disconnected from Interactive Brokers.")
else:
    print("\n  ℹ Simulation mode — no orders placed.")
    print("    To execute on paper trading: run_bot(execute_trades=True)")